In [3]:
import pandas as pd

BLACK_BELT_COUNTIES = [
    'Sumter', 'Choctaw', 'Greene', 'Hale', 'Marengo', 'Perry',
    'Dallas', 'Wilcox', 'Lowndes', 'Butler', 'Crenshaw',
    'Montgomery', 'Pike', 'Bullock', 'Macon', 'Barbour', 'Russell'
]

def build_dataset(sos_filepath, cvap_filepath, l2_filepath, cvap_black, cvap_white, cvap_hisp, output_name):
    # Load RDH file for FIPS code lookup, pad to 5 digits with leading zero
    l2 = pd.read_csv(l2_filepath)
    l2['county_name'] = l2['countyname'].str.strip().str.title()
    l2['fips_code'] = l2['countyfips'].astype(str).str.zfill(5)
    fips_lookup = l2[l2['county_name'].isin(BLACK_BELT_COUNTIES)][['county_name', 'fips_code']]

    # Load SoS race file (already cleaned with lowercase column names)
    sos = pd.read_excel(sos_filepath)
    sos['county_name'] = sos['county'].astype(str).str.strip().str.title()
    sos = sos[sos['county_name'].isin(BLACK_BELT_COUNTIES)].copy()
    sos = sos.rename(columns={
        'black_ballots':    'voted_black',
        'white_ballots':    'voted_white',
        'hispanic_ballots': 'voted_latinx',
    })[['county_name', 'voted_black', 'voted_white', 'voted_latinx']]

    # Load CVAP file for eligible population
    cvap = pd.read_csv(cvap_filepath)
    cvap['fips_code'] = cvap['GEOID'].astype(str).str.zfill(5)
    fips_set = set(fips_lookup['fips_code'])
    cvap = cvap[cvap['fips_code'].isin(fips_set)].copy()
    cvap = cvap.rename(columns={
        cvap_black: 'eligible_black',
        cvap_white: 'eligible_white',
        cvap_hisp:  'eligible_latinx',
    })[['fips_code', 'eligible_black', 'eligible_white', 'eligible_latinx']]

    # Merge FIPS lookup with SoS turnout and CVAP eligible
    df = fips_lookup.merge(sos, on='county_name', how='inner')
    df = df.merge(cvap, on='fips_code', how='inner')

    # Calculate missing voters and missing voters rate per race
    for race in ['black', 'white', 'latinx']:
        df[f'missing_voters_{race}'] = (
            df[f'eligible_{race}'] - df[f'voted_{race}']
        ).apply(lambda x: max(0, x))

        df[f'missing_voters_rate_{race}'] = (
            df[f'missing_voters_{race}'] / df[f'eligible_{race}']
        ).round(4)

    # Calculate the equity gap between Black and White non-participation rates
    # Positive value means Black non-participation is higher than White
    df['not_voted_gap'] = (df['missing_voters_rate_black'] - df['missing_voters_rate_white']).round(4)

    # Reorder columns for final output
    df = df[[
        'county_name', 'fips_code',
        'eligible_black',  'voted_black',  'missing_voters_black',  'missing_voters_rate_black',
        'eligible_white',  'voted_white',  'missing_voters_white',  'missing_voters_rate_white',
        'eligible_latinx', 'voted_latinx', 'missing_voters_latinx', 'missing_voters_rate_latinx',
        'not_voted_gap'
    ]]

    # Convert any missing values to null per standardization document
    df = df.where(df.notna(), other='null')

    # Sort alphabetically by county and reset index
    df = df.sort_values('county_name').reset_index(drop=True)

    df.to_csv(output_name, index=False)
    print(f"Created: {output_name}")
    return df

df_2024 = build_dataset(
    sos_filepath  = '../data_storage/raw_data/voter_data/2024_race_gen_election.xlsx',
    cvap_filepath = '../data_storage/raw_data/eligible_data/al_cvap_2024_cnty.csv',
    l2_filepath   = '../data_storage/raw_data/voter_data/AL_l2_2024_gen_stats_2020county.csv',
    cvap_black    = 'CVAP_BLA24',
    cvap_white    = 'CVAP_WHT24',
    cvap_hisp     = 'CVAP_HSP24',
    output_name   = '../data_storage/clean_data/eligiblepop_2024_general.csv'
)

df_2020 = build_dataset(
    sos_filepath  = '../data_storage/raw_data/voter_data/2020_race_gen_election.xlsx',
    cvap_filepath = '../data_storage/raw_data/eligible_data/al_cvap_2020_cnty.csv',
    l2_filepath   = '../data_storage/raw_data/voter_data/AL_l2_2024_gen_stats_2020county.csv',
    cvap_black    = 'CVAP_BLK20',
    cvap_white    = 'CVAP_WHT20',
    cvap_hisp     = 'CVAP_HSP20',
    output_name   = '../data_storage/clean_data/eligiblepop_2020_general.csv'
)

df_2024

Created: ../data_storage/clean_data/eligiblepop_2024_general.csv
Created: ../data_storage/clean_data/eligiblepop_2020_general.csv


,county_name,fips_code,eligible_black,voted_black,missing_voters_black,missing_voters_rate_black,eligible_white,voted_white,missing_voters_white,missing_voters_rate_white,eligible_latinx,voted_latinx,missing_voters_latinx,missing_voters_rate_latinx,not_voted_gap
0,Barbour,01005,8929,3773,5156,0.5774,9301,5979,3322,0.3572,444,50,394,0.8874,0.2202
1,Bullock,01011,5518,2966,2552,0.4625,1949,1099,850,0.4361,186,18,168,0.9032,0.0264
2,Butler,01013,6207,2998,3209,0.5170,7580,5019,2561,0.3379,164,18,146,0.8902,0.1791
3,Choctaw,01023,3722,2435,1287,0.3458,5637,4180,1457,0.2585,85,8,77,0.9059,0.0873
4,Crenshaw,01041,2425,1272,1153,0.4755,7218,5139,2079,0.2880,123,17,106,0.8618,0.1875
5,Dallas,01047,19033,10418,8615,0.4526,8106,5191,2915,0.3596,165,13,152,0.9212,0.0930
6,Greene,01063,4645,3175,1470,0.3165,1116,835,281,0.2518,4,12,0,0.0000,0.0647
7,Hale,01065,6158,3803,2355,0.3824,4597,3465,1132,0.2462,105,17,88,0.8381,0.1362
8,Lowndes,01085,5561,3916,1645,0.2958,2022,1751,271,0.1340,15,12,3,0.2000,0.1618
9,Macon,01087,12121,6097,6024,0.4970,2898,1693,1205,0.4158,195,26,169,0.8667,0.0812
